# YouTube lecture → deck package (Colab-first, v0.1)

Same split as Heme SH: **Colab does transcript + upload**, agent gates later.

**Preferred (no cookies):** timed **captions** → segments with start/end.  
**Fallback:** audio-only yt-dlp + Whisper (auto recompress / 10-min chunks if >24MB). Frames optional (`SKIP_FRAMES = True`).

If bot-blocked: **Runtime → Disconnect and delete runtime → reconnect** (new IP), then rerun. Cipriani worked that way without cookies.

| Setting | Value |
|---------|--------|
| Playback | YouTube `&t=NNNs` |
| Sidecar | `gs://pathology_hub/02_normalized/lectures/deck_packages/<package_id>/` |
| Index grain | empty until agent semantic gate |

Run cells in order.


In [ ]:
# Cell 1 — auth, installs, config
!pip -q install -U yt-dlp openai google-cloud-storage youtube-transcript-api

from google.colab import auth, userdata
auth.authenticate_user()

import json, os, re, shutil, subprocess, tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import parse_qs, urlparse
from urllib.request import urlopen

from google.cloud import storage
from openai import OpenAI

PROJECT = "pathology-annotation-project"
HUB_BUCKET = "pathology_hub"
VIDEO_BUCKET = "pathology-hub-0"
ASSET_PREFIX = "_asset_library/lectures/"

# --- edit these ---
YOUTUBE_URL = "https://www.youtube.com/watch?v=rCdaaTDesPQ"  # Damron Breast board review
ROOT = "Breast"
FRAME_EVERY_SEC = 15.0
SKIP_FRAMES = True
PREFER_CAPTIONS = True      # captions first; Whisper only if needed
UPLOAD_FRAMES_ASSETS = False
PACKAGE_ID_OVERRIDE = None  # e.g. "yt_damron_breast_board_review_v0_1"
WHISPER_MAX_MB = 24.0       # API soft limit ~25MB
WHISPER_CHUNK_SEC = 600     # 10-min chunks if still too large after recompress
# ------------------

# Colab secrets: prefer OPEN_AI_KEY_01 (Heme pattern), then OPENAI_API_KEY
def load_openai_key():
    for name in ("OPEN_AI_KEY_01", "OPENAI_API_KEY", "OPEN_AI_KEY"):
        try:
            k = userdata.get(name)
            if k:
                print("OpenAI key from Colab secret:", name)
                return k
        except Exception:
            pass
    k = os.environ.get("OPENAI_API_KEY")
    assert k, "Set Colab secret OPEN_AI_KEY_01 (or OPENAI_API_KEY)"
    return k

os.environ["OPENAI_API_KEY"] = load_openai_key()
oai = OpenAI()
gcs = storage.Client(project=PROJECT)
hub = gcs.bucket(HUB_BUCKET)
assets = gcs.bucket(VIDEO_BUCKET)

WORK = Path("/content/yt_ingest_work")
WORK.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def slugify(text: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9]+", "_", text.strip()).strip("_").lower()
    return s or "lecture"

def extract_youtube_id(url: str) -> str:
    u = urlparse(url)
    if u.netloc in {"youtu.be"}:
        return u.path.strip("/").split("/")[0]
    qs = parse_qs(u.query)
    if "v" in qs and qs["v"]:
        return qs["v"][0]
    m = re.search(r"(?:embed|shorts|live)/([A-Za-z0-9_-]{6,})", url)
    if m:
        return m.group(1)
    raise ValueError(url)

def youtube_watch_url(video_id: str) -> str:
    return f"https://www.youtube.com/watch?v={video_id}"

def make_youtube_time_url(video_id: str, start) -> Optional[str]:
    try:
        s = int(float(start))
    except (TypeError, ValueError):
        return None
    return f"https://www.youtube.com/watch?v={video_id}&t={max(0, s)}s"

def run(cmd):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True, capture_output=True, text=True)

VIDEO_ID = extract_youtube_id(YOUTUBE_URL)
YT_URL = youtube_watch_url(VIDEO_ID)
print("VIDEO_ID", VIDEO_ID)
print("WORK", WORK)


In [ ]:
# Cell 2 — metadata + captions-first (no cookies) OR audio-only Whisper fallback
# If bot-blocked: Runtime → Disconnect and delete runtime → reconnect, then rerun from Cell 1.

def fetch_meta(url: str) -> dict:
    try:
        proc = run(["yt-dlp", "--no-playlist", "--dump-single-json", "--skip-download", url])
        return json.loads(proc.stdout)
    except subprocess.CalledProcessError as exc:
        print("yt-dlp meta failed; oEmbed fallback\n", (exc.stderr or "")[-500:])
        oe = json.loads(urlopen(f"https://www.youtube.com/oembed?url={url}&format=json", timeout=30).read())
        return {
            "id": VIDEO_ID,
            "title": oe.get("title"),
            "uploader": oe.get("author_name"),
            "channel": oe.get("author_name"),
            "duration": None,
            "_meta_source": "oembed_fallback",
        }

def fetch_caption_segments(video_id: str) -> list:
    """Timed captions — no media download, no cookies when Colab IP is clean."""
    from youtube_transcript_api import YouTubeTranscriptApi
    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id)
    out = []
    for i, s in enumerate(fetched):
        if hasattr(s, "text"):
            text = (s.text or "").strip()
            start = float(s.start or 0.0)
            dur = float(getattr(s, "duration", 0.0) or 0.0)
        else:
            text = (s.get("text") or "").strip()
            start = float(s.get("start") or 0.0)
            dur = float(s.get("duration") or 0.0)
        if not text:
            continue
        out.append({"id": i, "start": start, "end": start + max(dur, 0.01), "text": text})
    return out

meta = fetch_meta(YOUTUBE_URL)
TITLE = meta.get("title") or f"YouTube {VIDEO_ID}"
UPLOADER = meta.get("uploader") or meta.get("channel") or ""
DURATION = meta.get("duration")
print(json.dumps({"title": TITLE, "uploader": UPLOADER, "duration": DURATION}, indent=2))

SEGMENTS = []
TRANSCRIPT_SOURCE = None
AUDIO = None
VIDEO = None

if PREFER_CAPTIONS:
    try:
        SEGMENTS = fetch_caption_segments(VIDEO_ID)
        TRANSCRIPT_SOURCE = "youtube_captions"
        print(f"captions OK segments={len(SEGMENTS)}")
        print("sample:", SEGMENTS[0] if SEGMENTS else None)
    except Exception as e:
        print("captions failed (will try audio+Whisper):", type(e).__name__, e)

if not SEGMENTS:
    audio_tmpl = str(WORK / "audio.%(ext)s")
    run([
        "yt-dlp", "--no-playlist",
        "-f", "bestaudio[ext=m4a]/bestaudio/best",
        "-o", audio_tmpl,
        "--extract-audio", "--audio-format", "mp3", "--audio-quality", "0",
        YOUTUBE_URL,
    ])
    AUDIO = next(p for p in WORK.glob("audio.*") if p.suffix.lower() in {".mp3", ".m4a", ".webm", ".opus", ".wav"})
    print("AUDIO", AUDIO, f"{AUDIO.stat().st_size/1e6:.1f} MB")
    if AUDIO.stat().st_size > 24.5e6:
        small = WORK / "audio_64k.mp3"
        run(["ffmpeg", "-y", "-i", str(AUDIO), "-b:a", "64k", str(small)])
        AUDIO = small
        print("recompressed", AUDIO, f"{AUDIO.stat().st_size/1e6:.1f} MB")

if DURATION is None and AUDIO is not None:
    try:
        proc = run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                    "-of", "default=noprint_wrappers=1:nokey=1", str(AUDIO)])
        DURATION = float(proc.stdout.strip())
    except Exception as e:
        print("WARN duration probe", e)

if not SKIP_FRAMES:
    try:
        run([
            "yt-dlp", "--no-playlist",
            "-f", "bv*[height<=480]+ba/b[height<=480]/worst",
            "-o", str(WORK / "video.%(ext)s"),
            "--merge-output-format", "mp4",
            YOUTUBE_URL,
        ])
        VIDEO = next((p for p in WORK.glob("video.*") if p.suffix.lower() in {".mp4", ".mkv", ".webm"}), None)
        print("VIDEO", VIDEO)
    except subprocess.CalledProcessError as exc:
        print("WARN video download failed; continuing without frames", (exc.stderr or "")[-400:])


In [ ]:
# Cell 3 — Whisper only if captions did not populate SEGMENTS
# Handles >25MB audio: recompress bitrate, then time-chunk with ffmpeg if still large.

def whisper_file(path: Path) -> list:
    with path.open("rb") as f:
        resp = oai.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )
    data = resp.model_dump() if hasattr(resp, "model_dump") else dict(resp)
    out = []
    for i, s in enumerate(data.get("segments") or []):
        text = (s.get("text") or "").strip()
        if not text:
            continue
        out.append({
            "id": i,
            "start": float(s.get("start") or 0.0),
            "end": float(s.get("end") or s.get("start") or 0.0),
            "text": text,
        })
    return out


def prepare_audio_under_limit(audio: Path, max_mb: float = WHISPER_MAX_MB) -> Path:
    size_mb = audio.stat().st_size / 1e6
    if size_mb <= max_mb:
        return audio
    print(f"audio {size_mb:.1f} MB > {max_mb} MB — recompressing")
    for br in ("64k", "48k", "32k", "24k"):
        out = WORK / f"audio_{br}.mp3"
        run(["ffmpeg", "-y", "-i", str(audio), "-ac", "1", "-ar", "16000", "-b:a", br, str(out)])
        sm = out.stat().st_size / 1e6
        print(f"  {br} -> {sm:.1f} MB")
        if sm <= max_mb:
            return out
    return out  # smallest attempt; may still need chunking


def whisper_chunked(audio: Path, chunk_sec: int = WHISPER_CHUNK_SEC, max_mb: float = WHISPER_MAX_MB) -> list:
    """Split long/large audio into time windows, Whisper each, offset timestamps."""
    chunk_dir = WORK / "whisper_chunks"
    if chunk_dir.exists():
        shutil.rmtree(chunk_dir)
    chunk_dir.mkdir(parents=True)
    pattern = str(chunk_dir / "chunk_%03d.mp3")
    # segment muxer: fixed-duration chunks
    run([
        "ffmpeg", "-y", "-i", str(audio),
        "-f", "segment", "-segment_time", str(int(chunk_sec)),
        "-reset_timestamps", "1",
        "-ac", "1", "-ar", "16000", "-b:a", "32k",
        pattern,
    ])
    parts = sorted(chunk_dir.glob("chunk_*.mp3"))
    print(f"whisper chunks: {len(parts)} x ~{chunk_sec}s")
    all_segs = []
    for i, part in enumerate(parts):
        sm = part.stat().st_size / 1e6
        if sm > max_mb:
            # emergency further split
            print(f"  chunk {i} still {sm:.1f} MB — splitting again")
            sub = whisper_chunked(part, chunk_sec=max(chunk_sec // 2, 120), max_mb=max_mb)
            offset = i * float(chunk_sec)
            for s in sub:
                s["start"] += offset
                s["end"] += offset
                s["id"] = len(all_segs)
                all_segs.append(s)
            continue
        print(f"  whisper chunk {i}: {part.name} ({sm:.1f} MB)")
        segs = whisper_file(part)
        offset = i * float(chunk_sec)
        for s in segs:
            s["start"] += offset
            s["end"] += offset
            s["id"] = len(all_segs)
            all_segs.append(s)
    return all_segs


if SEGMENTS:
    print(f"skip Whisper; using {TRANSCRIPT_SOURCE} ({len(SEGMENTS)} segments)")
else:
    assert AUDIO is not None, "No captions and no audio — restart Colab runtime (new IP) or retry later"
    audio = prepare_audio_under_limit(AUDIO, WHISPER_MAX_MB)
    size_mb = audio.stat().st_size / 1e6
    if size_mb <= WHISPER_MAX_MB:
        print(f"Whisper single-file {audio.name} ({size_mb:.1f} MB)")
        SEGMENTS = whisper_file(audio)
    else:
        print(f"still {size_mb:.1f} MB after recompress — chunked Whisper")
        SEGMENTS = whisper_chunked(audio, WHISPER_CHUNK_SEC, WHISPER_MAX_MB)
    TRANSCRIPT_SOURCE = "whisper_audio"
    print("segments", len(SEGMENTS))
    print("sample:", SEGMENTS[0] if SEGMENTS else None)

assert SEGMENTS, "Empty transcript"
if DURATION is None and SEGMENTS:
    DURATION = float(SEGMENTS[-1]["end"])
print("TRANSCRIPT_SOURCE", TRANSCRIPT_SOURCE, "DURATION", DURATION)


In [ ]:
# Cell 4 — optional ffmpeg frames
FRAMES = []
frames_dir = WORK / "frame_imgs"
if VIDEO and not SKIP_FRAMES:
    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir(parents=True)
    fps = 1.0 / max(FRAME_EVERY_SEC, 1.0)
    run([
        "ffmpeg", "-y", "-i", str(VIDEO),
        "-vf", f"fps={fps}", "-q:v", "3",
        str(frames_dir / "slide_%04d.jpg"),
    ])
    for i, path in enumerate(sorted(frames_dir.glob("slide_*.jpg"))):
        FRAMES.append({
            "frame_index": i,
            "start_sec": i * FRAME_EVERY_SEC,
            "file": path.name,
            "local_path": path,
        })
print("frames", len(FRAMES))

In [ ]:
# Cell 5 — build local deck package (YouTube playback)
display_title = f"{TITLE} ({UPLOADER})" if UPLOADER else TITLE
PACKAGE_ID = PACKAGE_ID_OVERRIDE or (
    f"yt_{slugify(UPLOADER)[:24]}_{slugify(TITLE)[:48]}_{VIDEO_ID.lower()}_v0_1"
)
if len(PACKAGE_ID) > 90:
    PACKAGE_ID = f"yt_{VIDEO_ID.lower()}_v0_1"

PKG = WORK / "packages" / PACKAGE_ID
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True)
(PKG / "frames").mkdir(exist_ok=True)

video_url = YT_URL
seg_rows = []
for s in SEGMENTS:
    start, end = float(s["start"]), float(s["end"])
    seg_rows.append({
        "schema_version": "lecture_deck_segment.v0_1",
        "package_id": PACKAGE_ID,
        "segment_id": f"{PACKAGE_ID}::seg_{int(s['id']):05d}",
        "start_sec": start,
        "end_sec": end,
        "text": s["text"],
        "language": "en",
        "video_id": VIDEO_ID,
        "video_url": video_url,
        "video_time_url": make_youtube_time_url(VIDEO_ID, start),
        "raw_source_gcs_uri": None,
        "raw_source_join_basis": "youtube_watch_url",
        "youtube_url": YT_URL,
        "primary_tag": None,
        "tag_status": "untagged",
        "root": ROOT,
        "indexable": False,
        "source_format": "youtube_whisper_colab_v0",
    })

ASSET_STEM = re.sub(r"[^A-Za-z0-9_]+", "_", PACKAGE_ID.replace("_v0_1", ""))
frame_rows = []
for fr in FRAMES:
    start = float(fr["start_sec"])
    idx = int(fr["frame_index"])
    dest_name = f"{ASSET_STEM}_slide_{idx:04d}.jpg"
    shutil.copy2(fr["local_path"], PKG / "frames" / fr["file"])
    frame_rows.append({
        "schema_version": "lecture_deck_frame.v0_1",
        "package_id": PACKAGE_ID,
        "frame_index": idx,
        "start_sec": start,
        "file": f"frames/{fr['file']}",
        "image_path": f"{ASSET_STEM}/{dest_name}",
        "asset_gcs_uri": f"gs://{VIDEO_BUCKET}/{ASSET_PREFIX}{ASSET_STEM}/{dest_name}",
        "video_id": VIDEO_ID,
        "video_url": video_url,
        "video_time_url": make_youtube_time_url(VIDEO_ID, start),
        "raw_source_join_basis": "youtube_watch_url",
        "youtube_url": YT_URL,
    })

manifest = {
    "schema_version": "lecture_deck_package.v0_1",
    "package_id": PACKAGE_ID,
    "title": display_title,
    "root": ROOT,
    "source_format": "youtube_ingest_colab_v0_1",
    "youtube_url": YT_URL,
    "youtube_video_id": VIDEO_ID,
    "video_file_declared": None,
    "duration_seconds": float(DURATION) if DURATION is not None else None,
    "video_id": VIDEO_ID,
    "raw_source_gcs_uri": None,
    "video_url": video_url,
    "raw_source_join_basis": "youtube_watch_url",
    "playback": "youtube",
    "counts": {
        "segments": len(seg_rows),
        "frames": len(frame_rows),
        "segments_with_video_time_url": sum(1 for s in seg_rows if s.get("video_time_url")),
        "frames_with_video_time_url": sum(1 for f in frame_rows if f.get("video_time_url")),
        "canonical_mp4_present": False,
    },
    "created_at_utc": utc_now(),
    "known_limitations": [
        "Playback is YouTube &t= (not GCS MP4 #t=).",
        "chunks_indexable.jsonl empty until agent semantic gate.",
        "No claim of FAISS/API exposure from this notebook.",
    ],
    "source_url": YOUTUBE_URL,
}

(PKG / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
with (PKG / "segments.jsonl").open("w") as f:
    for row in seg_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with (PKG / "frames.jsonl").open("w") as f:
    for row in frame_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
(PKG / "segments_indexable.jsonl").write_text("")
(PKG / "chunks_indexable.jsonl").write_text("")

local_audit = {
    "schema_version": "lecture_deck_youtube_colab_ingest_audit.v0_1",
    "created_at_utc": utc_now(),
    "package_id": PACKAGE_ID,
    "input_paths": [YOUTUBE_URL],
    "output_paths": [str(PKG)],
    "counts": manifest["counts"],
    "join": {"video_url": video_url, "raw_source_join_basis": "youtube_watch_url"},
    "known_limitations": manifest["known_limitations"],
}
(PKG / "audit.json").write_text(json.dumps(local_audit, indent=2) + "\n")

print("PACKAGE_ID", PACKAGE_ID)
print("PKG", PKG)
print("segments", len(seg_rows), "frames", len(frame_rows))

In [ ]:
# Cell 6 — upload sidecar (+ optional frame assets) + audit
DRY_RUN = False   # True = print destinations only
SKIP_EXISTING_FRAMES = True

uploaded = []
prefix = f"02_normalized/lectures/deck_packages/{PACKAGE_ID}/"
for name in (
    "manifest.json", "segments.jsonl", "frames.jsonl", "audit.json",
    "segments_indexable.jsonl", "chunks_indexable.jsonl",
):
    path = PKG / name
    if not path.is_file():
        continue
    dest = prefix + name
    if DRY_RUN:
        print("DRY", f"gs://{HUB_BUCKET}/{dest}")
    else:
        hub.blob(dest).upload_from_filename(str(path))
        print("uploaded", f"gs://{HUB_BUCKET}/{dest}")
    uploaded.append(f"gs://{HUB_BUCKET}/{dest}")

frame_upload_count = 0
frame_skip_count = 0
if UPLOAD_FRAMES_ASSETS and FRAMES:
    for fr in FRAMES:
        idx = int(fr["frame_index"])
        dest_name = f"{ASSET_STEM}_slide_{idx:04d}.jpg"
        dest_key = f"{ASSET_PREFIX}{ASSET_STEM}/{dest_name}"
        blob = assets.blob(dest_key)
        if SKIP_EXISTING_FRAMES and blob.exists():
            frame_skip_count += 1
            continue
        if DRY_RUN:
            print("DRY", f"gs://{VIDEO_BUCKET}/{dest_key}")
        else:
            blob.upload_from_filename(str(fr["local_path"]), content_type="image/jpeg")
            frame_upload_count += 1

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
gcs_audit = {
    "schema_version": "lecture_deck_youtube_colab_upload_audit.v0_1",
    "created_at_utc": utc_now(),
    "package_id": PACKAGE_ID,
    "input_paths": [YOUTUBE_URL, str(PKG)],
    "output_paths": [
        f"gs://{HUB_BUCKET}/{prefix}",
        f"gs://{VIDEO_BUCKET}/{ASSET_PREFIX}{ASSET_STEM}/",
    ],
    "counts": {
        **manifest["counts"],
        "sidecar_files_uploaded": len(uploaded),
        "frames_uploaded": frame_upload_count,
        "frames_skipped_existing": frame_skip_count,
        "dry_run": DRY_RUN,
    },
    "uploaded": uploaded,
    "known_limitations": [
        "Sidecar only; not semantic-gated; not in lecture FAISS until agent rebuild.",
        "Playback remains YouTube; no source_videos MP4 uploaded by design (approach B).",
    ],
}
audit_key = f"06_audits/lectures/deck_packages/youtube_colab_ingest_{stamp}/audit.json"
if not DRY_RUN:
    hub.blob(audit_key).upload_from_string(json.dumps(gcs_audit, indent=2) + "\n", content_type="application/json")
print("audit =>", f"gs://{HUB_BUCKET}/{audit_key}")
print("PACKAGE_ID for agent:", PACKAGE_ID)
print("Tell the agent: gate BST package", PACKAGE_ID, "then vector rebuild + Cloud Run refresh")

## After Colab succeeds

Reply in the agent chat with the printed `PACKAGE_ID` (or just **gate Cipriani**).

Agent will:

1. Download sidecar → run `build_lecture_deck_semantic_indexable_chunks_v0_2.py --root BST`
2. Re-upload `chunks_indexable.jsonl`
3. `build_lecture_vector_from_deck_packages_v0_1.py --upload --promote-live`
4. Remind you to refresh Cloud Run (`LECTURE_MANIFEST_REFRESH_TS`)

Handoff: `docs/COLAB_TODO_YOUTUBE_LECTURE_INGEST.md`